# Demo 08 — Body-based routing (pre-provisioned API key)

Same BBR walkthrough as **`BodyBasedRouting.ipynb`**, but you paste an existing **`sk-oai-*`** key (no OpenShift / key-creation steps).

**Prereq:** Demo 08 applied; key bound to **`demo08-hybrid-catalog`**. Python 3.9+ stdlib only.


## Demo quick swap

| Variable | Effect |
|----------|--------|
| `DEMO_MAAS_BASE` | Non-empty → overrides `MAAS_BASE`. |
| `DEMO_API_KEY` | Non-empty → overrides `MAAS_API_KEY` / `API_KEY`. |


In [ ]:
DEMO_MAAS_BASE = ""
DEMO_API_KEY = ""


## Setup


In [ ]:
import json
import os
import ssl
import urllib.error
import urllib.request
from typing import Any, Dict, Optional

_mb = globals().get("DEMO_MAAS_BASE", "")
if isinstance(_mb, str) and _mb.strip():
    MAAS_BASE = _mb.strip().rstrip("/")
else:
    MAAS_BASE = os.environ.get("MAAS_BASE", "https://maas.YOUR_DOMAIN_HERE").strip().rstrip("/")

_ak = globals().get("DEMO_API_KEY", "")
if isinstance(_ak, str) and _ak.strip():
    API_KEY = _ak.strip()
else:
    API_KEY = (os.environ.get("MAAS_API_KEY") or os.environ.get("API_KEY") or "").strip()

VERIFY_TLS = os.environ.get("VERIFY_TLS", "").lower() in ("1", "true", "yes")
MODELS_URL = f"{MAAS_BASE}/maas-api/v1/models"
BBR_CHAT_URL = f"{MAAS_BASE}/v1/chat/completions"

if not API_KEY:
    raise SystemExit("Set DEMO_API_KEY or MAAS_API_KEY / API_KEY.")


def http_json(method: str, url: str, *, token: Optional[str] = None, data: Optional[Dict[str, Any]] = None):
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    body = json.dumps(data).encode("utf-8") if data is not None else None
    ctx = ssl.create_default_context()
    if not VERIFY_TLS:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=120) as resp:
            raw = resp.read().decode("utf-8")
            return resp.status, json.loads(raw) if raw else {}
    except urllib.error.HTTPError as e:
        err_body = e.read().decode("utf-8", errors="replace")
        try:
            parsed = json.loads(err_body) if err_body else {}
        except json.JSONDecodeError:
            parsed = {"_raw": err_body}
        raise RuntimeError(f"HTTP {e.code}: {parsed}") from None


print("MAAS_BASE    :", MAAS_BASE)
print("BBR_CHAT_URL :", BBR_CHAT_URL)
print("VERIFY_TLS   :", VERIFY_TLS)
print("API key set  :", bool(API_KEY))


## Discover models + BBR chat


In [ ]:
_, models_body = http_json("GET", MODELS_URL, token=API_KEY)
data = models_body.get("data") or []
if not data:
    raise SystemExit("No models in catalog.")


def _pick(pred):
    for m in data:
        mid = (m.get("id") or m.get("name") or "")
        if pred(mid.lower()):
            return m
    return None


granite = _pick(lambda s: "granite" in s) or data[0]
sim_chat = _pick(lambda s: "sim-chat" in s and "sim-chat-2" not in s)
GRANITE_ID = granite.get("id") or granite.get("name")
GRANITE_URL = (granite.get("url") or "").rstrip("/")
SIM_CHAT_ID = (sim_chat.get("id") or sim_chat.get("name")) if sim_chat else None

print("GRANITE_ID:", GRANITE_ID)
print("SIM_CHAT_ID:", SIM_CHAT_ID or "(missing)")
for m in data:
    print(" -", m.get("id") or m.get("name"), "→", m.get("url"))


def chat_completion(*, url: str, model: str, user_message: str, max_tokens: int = 64):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": user_message}],
        "max_tokens": max_tokens,
    }
    status, body = http_json("POST", url, token=API_KEY, data=payload)
    choices = body.get("choices") or []
    msg = (choices[0].get("message") or {}) if choices and isinstance(choices[0], dict) else {}
    print("HTTP", status, "|", model)
    print("Assistant:", msg.get("content") or "(empty)")
    if body.get("usage"):
        print("usage:", body["usage"])
    print()
    return status, body


print("=== BBR Granite ===")
chat_completion(url=BBR_CHAT_URL, model=GRANITE_ID, user_message="Say hello in one sentence.")

if SIM_CHAT_ID:
    print("=== BBR sim-chat (llm-katan) ===")
    try:
        chat_completion(url=BBR_CHAT_URL, model=SIM_CHAT_ID, user_message="Say hello in one sentence.")
    except RuntimeError as e:
        print("External failed:", e)

if GRANITE_URL:
    print("=== Path-based Granite ===")
    chat_completion(
        url=f"{GRANITE_URL}/v1/chat/completions",
        model=GRANITE_ID,
        user_message="Say hello in one sentence.",
    )
